# SAR Scalar Speckle Filters Tutorial

This notebook demonstrates scalar (single-channel) speckle filters from GRDL on SAR SLC imagery.
Filters operate independently on each polarization channel; for joint polarimetric filtering see
`sar_matrix_filters_tutorial.ipynb`.

**Algorithms demonstrated:**
1. `LeeFilter` — classic Lee adaptive filter (amplitude domain)
2. `ComplexLeeFilter` — Lee filter on complex (SLC) data preserving phase
3. `LeeSigmaFilter` — improved Lee sigma filter (placeholder)

**Data:** NISAR L1 RSLC (HH channel) or synthetic fallback.

## Theory — SAR Speckle and the Lee Filter

SAR speckle arises from the coherent addition of returns from many unresolved scatterers within a
resolution cell. For a single-look image the intensity $I$ follows an exponential distribution:

$$I = \sigma^0 \cdot v, \quad v \sim \text{Exp}(1)$$

The Lee MMSE filter estimates the true reflectivity $\hat{I}$ as:

$$\hat{I} = \bar{I} + b (I - \bar{I}), \quad b = \frac{c_v^2 - c_u^2}{c_v^2 (1 + c_u^2)}$$

where $\bar{I}$ is the local window mean, $c_v = \text{std}(I)/\bar{I}$ is the local coefficient of
variation, and $c_u = 1/\sqrt{\mathrm{ENL}}$ is the noise coefficient from the number of looks.

The **ComplexLeeFilter** extends this to complex SLC data, filtering amplitude while preserving the
interferometric phase:

$$\hat{s} = \bar{s} + b_c (s - \bar{s}), \quad b_c = \frac{c_v^2 - c_u^2}{c_v^2 (1 + c_u^2)}$$

### Effective Number of Looks (ENL)

ENL measures the degree of speckle averaging.  For fully-developed speckle:

$$\mathrm{ENL} = \left(\frac{\mu_I}{\sigma_I}\right)^2$$

Successful speckle filtering should increase the ENL without blurring strong scatterers.

**References:**
- Lee, J.-S. (1980). *Digital image enhancement and noise filtering by use of local statistics.*
  IEEE Trans. Pattern Analysis and Machine Intelligence, 2(2), 165–168.
- Lee, J.-S. (1994). *Speckle suppression and analysis for synthetic aperture radar images.*
  Optical Engineering, 25(5), 255636.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
import gc

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from grdl.IO.sar.nisar import NISARReader
from grdl.image_processing.filters import LeeFilter, ComplexLeeFilter, LeeSigmaFilter, EnhancedLeeFilter
from grdk.viewers.dual_viewer import DualGeoViewer

%gui qt6
print('Imports OK')


In [ ]:
# =============================================================================
# Configuration
# =============================================================================
nisar_file = Path(
    '/data/sar/slc/nisar/l1_rslc/'
    '20260105T055924_20260105T055931/'
    'NISAR_L1_PR_RSLC_009_116_D_054_2005_QPDH_A_20260105T055924_20260105T055931'
    '_X05010_N_P_J_001.h5'
)
frequency    = 'A'
polarization = 'HH'
chip_size    = 1024   # pixels (square); None = full image

# Filter parameters
kernel_size = 7    # window side length (odd)
enl         = 1.0  # number of looks (0 = auto-estimate)

In [ ]:
# =============================================================================
# Load NISAR HH or generate synthetic single-pol SLC
# =============================================================================
def _synthetic_slc(rows=512, cols=512, seed=42):
    """Rayleigh-distributed complex SLC with simulated point targets."""
    rng = np.random.default_rng(seed)
    amp = rng.rayleigh(scale=1.0, size=(rows, cols)).astype(np.float32)
    phase = rng.uniform(0, 2 * np.pi, size=(rows, cols)).astype(np.float32)
    slc = amp * np.exp(1j * phase)
    # add three bright point targets
    for r_frac, c_frac, a in [(0.25, 0.25, 15.0), (0.5, 0.5, 20.0), (0.75, 0.75, 15.0)]:
        slc[int(r_frac * rows), int(c_frac * cols)] += a
    return slc.astype(np.complex64)

if nisar_file.exists():
    reader = NISARReader(nisar_file, frequency=frequency, polarization=polarization)
    meta = reader.metadata
    N = chip_size or meta.rows
    r0 = max(0, (meta.rows - N) // 2)
    c0 = max(0, (meta.cols - N) // 2)
    slc = reader.read_chip(r0, r0 + N, c0, c0 + N)
    reader.close()
    print(f'NISAR {polarization}: {slc.shape}, dtype={slc.dtype}')
else:
    N = chip_size or 512
    slc = _synthetic_slc(rows=N, cols=N)
    print(f'Synthetic SLC: {slc.shape} (no NISAR file found at {nisar_file})')

In [ ]:
# =============================================================================
# Display helpers
# =============================================================================
def to_db_display(data, pct=(2, 98)):
    """Convert amplitude^2 to dB, percentile-stretch to [0,1]."""
    power = np.abs(data) ** 2
    db = 10.0 * np.log10(np.maximum(power, 1e-10))
    lo, hi = np.percentile(db, pct)
    out = np.clip((db - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)
    return np.stack([out, out, out], axis=0)   # (3, rows, cols) grayscale

def estimate_enl(data):
    """Estimate ENL from intensity image (mean^2 / variance)."""
    intensity = np.abs(data) ** 2
    mu = float(np.mean(intensity))
    var = float(np.var(intensity))
    return mu ** 2 / max(var, 1e-12)

print(f'ENL (raw):  {estimate_enl(slc):.2f}')

---
## 1. LeeFilter — Amplitude-Domain Adaptive Filter

The `LeeFilter` operates on intensity (or amplitude) images.  The local window statistics
drive a spatially adaptive MMSE estimate, reducing speckle in homogeneous regions while
preserving edges and point targets.

In [ ]:
# =============================================================================
# LeeFilter — operates on real-valued amplitude; convert complex SLC first
# =============================================================================
lf = LeeFilter(kernel_size=kernel_size, enl=enl)
slc_lee = lf.apply(np.abs(slc))   # |z|: real amplitude

print(f'LeeFilter kernel={kernel_size}, enl={enl}')
print(f'ENL before: {estimate_enl(slc):.2f}')
print(f'ENL after:  {estimate_enl(slc_lee):.2f}')

rgb_orig = to_db_display(slc)
rgb_lee  = to_db_display(slc_lee)


In [ ]:
# =============================================================================
# GRDK dual-pane viewer — LeeFilter
# =============================================================================
viewer_lee = DualGeoViewer()
viewer_lee.set_mode('dual')
viewer_lee.setWindowTitle(f'LeeFilter (k={kernel_size}) — Original (L) vs Filtered (R)')
viewer_lee.set_array(rgb_orig, pane=0)
viewer_lee.set_array(rgb_lee, pane=1)
viewer_lee.show()
print('Viewer: LeeFilter')

---
## 2. ComplexLeeFilter — Phase-Preserving SLC Filter

The `ComplexLeeFilter` extends Lee filtering to complex SLC data.  The MMSE weight $b$ is
computed from intensity statistics as before, but is applied to the complex samples,
so interferometric phase is preserved alongside speckle reduction.

Use this filter when the filtered output will feed an InSAR or PolInSAR processor that
requires coherent (complex) values.

In [ ]:
# =============================================================================
# ComplexLeeFilter
# =============================================================================
clf = ComplexLeeFilter(kernel_size=kernel_size, enl=enl)
slc_clf = clf.apply(slc)

print(f'ComplexLeeFilter kernel={kernel_size}, enl={enl}')
print(f'ENL before: {estimate_enl(slc):.2f}')
print(f'ENL after:  {estimate_enl(slc_clf):.2f}')

# Phase difference: should be near-zero in homogeneous areas
phase_diff = np.angle(slc_clf * np.conj(slc))
print(f'Phase difference std: {np.std(phase_diff):.4f} rad')

rgb_clf = to_db_display(slc_clf)

In [ ]:
# =============================================================================
# GRDK dual-pane viewer — ComplexLeeFilter
# =============================================================================
viewer_clf = DualGeoViewer()
viewer_clf.set_mode('dual')
viewer_clf.setWindowTitle(f'ComplexLeeFilter (k={kernel_size}) — Original (L) vs Filtered (R)')
viewer_clf.set_array(rgb_orig, pane=0)
viewer_clf.set_array(rgb_clf, pane=1)
viewer_clf.show()
print('Viewer: ComplexLeeFilter')

---
## 3. LeeSigmaFilter — Sigma-Interval Adaptive Filter

The Lee Sigma filter selects only those neighbours whose amplitude falls within the
$[(1-\sigma)/2, (1+\sigma)/2]$ quantiles of the $\Gamma(\text{ENL}, 1/\text{ENL})$
speckle distribution for each centre pixel.  Local MMSE statistics are computed
exclusively from these sigma-selected neighbours, avoiding contamination from
bright heterogeneous pixels or edges that would bias the standard Lee filter.

An amplitude-domain MMSE weight is then applied and the original phase is preserved
for complex SLC input — giving ``|LeeSigmaFilter(z)| == LeeSigmaFilter(|z|)``.


In [ ]:
# =============================================================================
# LeeSigmaFilter — sigma-interval adaptive speckle filter
# =============================================================================

# sigma=0.9: select neighbours whose intensity falls within the 5th–95th
# percentile of the Gamma(ENL, 1/ENL) speckle distribution.
lsf = LeeSigmaFilter(kernel_size=kernel_size, enl=enl, sigma=0.9)
slc_sigma = lsf.apply(slc)

print(f'LeeSigmaFilter kernel={kernel_size}, sigma=0.9, ENL={enl or "auto"}')
print(f'  ENL before: {estimate_enl(slc):.2f}')
print(f'  ENL after:  {estimate_enl(slc_sigma):.2f}')

rgb_sigma = to_db_display(slc_sigma)

In [ ]:
viewer_sigma = DualGeoViewer()
viewer_sigma.set_mode('dual')
viewer_sigma.setWindowTitle('LeeSigmaFilter — Original (L) vs Filtered (R)')
viewer_sigma.set_array(rgb_orig, pane=0)
viewer_sigma.set_array(rgb_sigma, pane=1)
viewer_sigma.show()

---
## 4. EnhancedLeeFilter — Hard-Threshold Adaptive Filter

The Enhanced Lee filter (Lopes et al., 1990) classifies each pixel into one of three regimes
based on the local amplitude coefficient of variation $c_i = \sigma_\text{local}(|z|) / \mu_\text{local}(|z|)$:

| Regime | Condition | Output |
| --- | --- | --- |
| Homogeneous | $c_i \le c_u = 1/\sqrt{\text{ENL}}$ | $\mu_\text{local}$ (full smoothing) |
| Mixed | $c_u < c_i < c_\max = \sqrt{1+2/\text{ENL}}$ | $\mu W + |z|(1-W)$ |
| Point target | $c_i \ge c_\max$ | $|z|$ (no smoothing) |

where $W = \exp\!\left(-\text{damp}\cdot(c_i - c_u)/(c_\max - c_i)\right)$.

Phase is preserved exactly for complex SLC input.


In [ ]:
# =============================================================================
# EnhancedLeeFilter — hard-threshold adaptive speckle filter
# =============================================================================

elf = EnhancedLeeFilter(kernel_size=kernel_size, enl=enl, damp=1.0)
slc_elf = elf.apply(slc)

print(f'EnhancedLeeFilter kernel={kernel_size}, damp=1.0, ENL={enl or "auto"}')
print(f'  ENL before: {estimate_enl(slc):.2f}')
print(f'  ENL after:  {estimate_enl(slc_elf):.2f}')

rgb_elf = to_db_display(slc_elf)


In [ ]:
viewer_elf = DualGeoViewer()
viewer_elf.set_mode('dual')
viewer_elf.setWindowTitle('EnhancedLeeFilter — Original (L) vs Filtered (R)')
viewer_elf.set_array(rgb_orig, pane=0)
viewer_elf.set_array(rgb_elf, pane=1)
viewer_elf.show()


---
## ENL Summary

The table below summarises the Effective Number of Looks before and after each filter,
measured over the full chip.  Higher ENL indicates more speckle reduction.

In [ ]:
# =============================================================================
# ENL summary table
# =============================================================================
results = {
    'Raw':              estimate_enl(slc),
    'LeeFilter':        estimate_enl(slc_lee),
    'ComplexLee':       estimate_enl(slc_clf),
    'LeeSigmaFilter':   estimate_enl(slc_sigma),
    'EnhancedLeeFilter': estimate_enl(slc_elf),
}

print(f'{"Filter":<22} {"ENL":>8}')
print('-' * 32)
for name, val in results.items():
    print(f'{name:<22} {val:>8.2f}')
